생성적 적대 신경망(Generative Adversarial Networks), 줄여서 'GAN(간)'은 딥러닝의 원리를 활용해 가상 이미지를 생성하는 알고리즘이다.  
가짜를 만들어 내는 파트를 '생성자(Generator)', 진위를 가려내는 파트를 '판별자(Discriminator)'라고 한다.

생성자(generator)는 가상의 이미지를 만들어 내는 공장이다.  
DCGAN은 생성자가 가짜 이미지를 만들 때 컨볼루션 신경망(CNN)을 이용한다. 하지만 컨볼루션 신경망과 달리 옵티마이저(optimizer)를 사용하는 최적화 과정이나 컴파일하는 과정이 없다는 것이다. 판별과 학습이 이곳 생성자에서 일어나는 것이 아니기 때문이다. 또한, 일부 매개변수를 삭제하는 풀링(pooling) 과정이 없는 대신 패딩(padding) 과정이 포함된다. 빈 곳을 채워서 같은 크기로 맞추는 패딩 과정이 등장하는 이유는 입력 크기와 출려 크기를 똑같이 맞추기 위해서이다. 패딩 과정을 통해 생성자가 만들어 내는 이미지의 크기를 조절해야 하는 이유는 판별자가 비교할 '진짜'와 똑같은 크기가 되어야 하기 때문이다.  
케라스 패딩 함수는 이러한 문제를 쉽게 처리할 수 있도록 도와준다. padding='same'이라는 설정을 통해 입력과 출력의 크기가 다를 경우 자동으로 크기를 확장해 주고, 확장된 공간에 0을 채워 넣을 수 있다.

DCGAN의 제안자들은 학습에 꼭 필요한 옵션들을 제시했는데, 그중 하나가 배치 정규화(Batch Normalization)이다. 배치 정규화란 입력 데이터의 평균이 0, 분산이 1이 되도록 재배치하는 것인데, 다음 층으로 입력될 값을 일정하게 재배치하는 역할을 한다.  
케라스는 이를 쉽게 적용할 수 있게끔 BatchNomalization() 함수를 제공한다.  
또한, 생성자의 활성화 함수로는 Relu() 함수를 쓰고 판별자로 넘겨주기 직전에는 tanh() 함수를 쓰고 있다. tanh() 함수를 쓰면 출력되는 값을 -1~1사이로 맞출 수 있다.

설명할 내용을 코드로 정리하면 다음과 같다.


```
generator = Sequential() # 모델 이름을 generator로 정하고 Sequential() 함수를 호출
generator.add(Dense(128*7*7, input_dim=100, activation=LeakyReLU(0.2)))
generator.add(BatchNormalization())
generator.add(Reshape((7, 7, 128)))
generator.add(UpSampling2D())
generator.add(Conv2D(64, kernel_size=5, padding='same'))
generator.add(BatchNomalization())
generator.add(Activation(LeakyReLU(0.2)))
generator.add(UpSampling2D())
generator.add(Conv2D(1, kernel_size=5, padding='same', activation='tanh'))
```





```
generator.add(Dense(128*7*7, input_dim=100, activation=LeakyReLU(0.2)))
```
128은 임의로 정한 노두의 수이다.  
input_dim=100은 100차원 크기의 랜덤 벡터를 준비해 집어넣으라는 의미이다.  
여기서 주의할 부분은 7*7이다. 이는 이미지의 최초의 크기를 의미한다.  
GAN에선는 기존에 사용하던 ReLu() 함수를 쓸 경우 학습이 불안정해지는 경우가 많아, ReLU()를 조금 변형한 LeakyReLU() 함수를 쓴다.
LeakyReLU() 함수는 ReLU() 함수에서 x값이 음수이면 무조건 0이 되어 뉴런들이 일찍 소실되는 단점을 보완하기 위해, 0 이하에서도 작은 값을 갖게 만드는 활성화 함수이다. 케라스 함수를 이용해 LeakyReLU(0.2) 형태로 설정하면 0보다 작을 경우 0.2를 곱하라는 의미이다.

데이터 배치를 정규 분포로 만드는 배치 정규화 코드이다.


```
generator.add(BatchNormalization())
```



다음 코드는 컨볼루션 레이어가 받아들일 수 있는 형태로 바꾸어 주는 코드이다.

```
generator.add(Reshape((7, 7, 128)))
```

다음 코드는 두 배씩 업(up)샘플링을 한 후 컨볼루션 과정을 처리하는 코드이다.


```
generator.add(UpSampling2D())
generator.add(Conv2D(64, kernel_size=5, padding='same'))
```

UpSampling2D() 함수는 이미지의 가로세로 크기를 두 배씩 늘려준다. 7x7이 레이어를 지나며 그 크기가 14x14가 되고, 레이어를 지나며 28x28이 되는 것이다.  
conv2D() 함수에서 padding='same' 조건을 넣으면 모자라는 부분은 자동으로 0이 채워진다.  
케라스는 Upsampling과 Conv2D를 합쳐 놓은 Conv2DTranspose() 함수도 제공한다.


한 번 더 컨볼루션 과정을 거친 후 판별자로 값을 넘길 준비를 마친다.  
활성화 함수는 tanh() 함수을 사용한다.
```
generator.add(Conv2D(1, kernel_size=5, padding='same', activation='tanh'))
```



In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, BatchNormalization, Activation, LeakyReLU, UpSampling2D, Conv2D

generator = Sequential() # 모델 이름을 generator로 정하고 Sequential() 함수를 호출
generator.add(Dense(128*7*7, input_dim=100, activation=LeakyReLU(0.2)))
generator.add(BatchNormalization())
generator.add(Reshape((7, 7, 128)))
generator.add(UpSampling2D())
generator.add(Conv2D(64, kernel_size=5, padding='same'))
generator.add(BatchNormalization())
generator.add(Activation(LeakyReLU(0.2)))
generator.add(UpSampling2D())
generator.add(Conv2D(1, kernel_size=5, padding='same', activation='tanh'))
generator.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 6272)              633472    
                                                                 
 batch_normalization (Batch  (None, 6272)              25088     
 Normalization)                                                  
                                                                 
 reshape (Reshape)           (None, 7, 7, 128)         0         
                                                                 
 up_sampling2d (UpSampling2  (None, 14, 14, 128)       0         
 D)                                                              
                                                                 
 conv2d (Conv2D)             (None, 14, 14, 64)        204864    
                                                                 
 batch_normalization_1 (Bat  (None, 14, 14, 64)        2